# SSAFY Recycling VQA — Qwen3-VL-8B-Instruct (Colab H100)

이 노트북은 **Colab H100** 기준 최상위권용 파이프라인이다.

핵심 구성:
- Qwen3-VL-8B-Instruct
- 4-bit QLoRA 학습
- shared adapter + count expert adapter
- dev pseudo label
- public TrashNet synthetic QA
- logprob 기반 4지선다 추론
- option permutation averaging
- EasyOCR 보조
- text-only prior

권장 실행 순서:
1. 설치 셀
2. 데이터 준비
3. 설정 셀 수정
4. 전체 코드 정의 셀
5. `main()` 실행


## 1) 환경 설치

Qwen3-VL 공식 카드 기준으로 최신 `transformers` 소스 설치가 안전하다.  
H100에서는 `flash_attention_2`가 잡히면 쓰고, 실패하면 자동으로 `sdpa`로 내려간다.


In [ ]:
!nvidia-smi
!pip -q install --upgrade pip
!pip -q install git+https://github.com/huggingface/transformers accelerate peft bitsandbytes datasets scikit-learn qwen-vl-utils easyocr opencv-python-headless sentencepiece ninja
!pip -q install flash-attn --no-build-isolation || true

In [ ]:
import torch, transformers, peft, datasets, sklearn
print("torch:", torch.__version__)
print("cuda :", torch.version.cuda)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2) 데이터 준비

대회 파일을 `/content` 아래에 두거나, 압축 해제 후 `train.csv`, `test.csv`, `dev.csv`가 보이는 폴더를 만들면 된다.

옵션:
- 그냥 Colab에 업로드
- Google Drive 마운트 후 사용


In [ ]:
# Optional: Google Drive mount
# from google.colab import drive
# drive.mount('/content/drive')

## 3) 설정 수정

필요한 항목만 바꿔라.
- `cfg.data_root`: 데이터 루트 폴더
- `cfg.use_trashnet`: public external data 사용 여부
- `cfg.use_taco_manual_hook`: 수동으로 받은 TACO COCO-format 폴더 사용 여부
- `cfg.force_retrain`: 기존 adapter 덮어쓰기 여부


In [ ]:

# -*- coding: utf-8 -*-
"""
SSAFY Recycling VQA - Qwen3-VL-8B-Instruct competition notebook/script
Environment preset: colab_h100

What this script does
---------------------
1) Loads train/test/dev competition data.
2) Builds a stronger training pool:
   - original train
   - high-confidence pseudo labels from dev majority vote
   - optional public TrashNet material-question synthetic data
   - optional public TACO hook (manual download path)
3) Fine-tunes Qwen3-VL-8B-Instruct with 4-bit QLoRA (shared adapter).
4) Fine-tunes an additional count expert adapter on count-focused samples.
5) Trains a fast text-only prior (TF-IDF + LogisticRegression).
6) Runs stable logprob-based multiple-choice inference with option-permutation averaging.
7) Optionally applies EasyOCR boosts for OCR/brand/volume questions.
8) Blends shared/count/text/OCR scores and writes submission.csv.

Notes
-----
- Highest single-model ceiling: Qwen3-VL-8B-Instruct.
- Stable local execution: use SDPA attention by default on local CUDA 13.
- H100 preset can use flash_attention_2 if installation succeeds; otherwise SDPA fallback.
"""

from __future__ import annotations

import os
import gc
import re
import json
import math
import time
import copy
import random
import warnings
import hashlib
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from datasets import load_dataset

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

# Qwen3-VL class name in latest Transformers
from transformers import Qwen3VLForConditionalGeneration

from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

# -----------------------------
# Configuration
# -----------------------------

@dataclass
class CFG:
    env_name: str = "colab_h100"
    base_model: str = "Qwen/Qwen3-VL-8B-Instruct"
    seed: int = 42

    # Paths
    data_root: Optional[str] = None
    work_dir: str = "./work_qwen3vl_colab_h100"
    submission_name: str = "submission_qwen3vl_colab_h100.csv"

    # Runtime
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_bf16: bool = True
    num_workers: int = 2
    pin_memory: bool = True

    # Attention / quantization
    attn_implementation: str = "flash_attention_2"  # "flash_attention_2" or "sdpa"
    train_load_in_4bit: bool = True
    infer_load_in_4bit: bool = False
    bnb_4bit_compute_dtype: str = "bfloat16"
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_use_double_quant: bool = True

    # Image handling (manual resize for stability)
    train_max_side: int = 1344
    infer_max_side: int = 1536
    heavy_max_side: int = 1792
    min_side: int = 448
    multiple_of: int = 28
    jpeg_quality_augment: bool = False

    # Training schedule
    do_train: bool = True
    do_train_shared: bool = True
    do_train_count_expert: bool = True
    force_retrain: bool = False

    shared_epochs: int = 2
    count_epochs: int = 1
    shared_train_batch_size: int = 2
    count_train_batch_size: int = 2
    gradient_accumulation_steps: int = 8
    max_grad_norm: float = 1.0
    lr_shared: float = 0.0002
    lr_count: float = 0.00015
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    logging_steps: int = 20
    save_every_epoch: bool = True

    # LoRA
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    lora_target_modules: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )

    # Data
    option_shuffle_prob: float = 0.9
    upsample_ocr_text: int = 3
    upsample_location: int = 2
    upsample_color: int = 2
    upsample_count: int = 1
    use_dev_pseudo: bool = True
    pseudo_min_conf_shared: float = 0.80
    pseudo_min_conf_count: float = 0.60

    # External public data
    use_trashnet: bool = True
    use_taco_manual_hook: bool = False
    taco_manual_dir: Optional[str] = None  # e.g. "/content/external/taco"

    # Text prior
    use_text_prior: bool = True
    tfidf_ngram_range: Tuple[int, int] = (2, 5)
    tfidf_min_df: int = 2
    tfidf_analyzer: str = "char_wb"
    text_prior_c: float = 4.0
    text_prior_max_iter: int = 2000

    # Inference
    do_infer: bool = True
    infer_batch_size: int = 2
    infer_num_permutations: int = 2
    do_heavy_second_pass: bool = True
    heavy_second_pass_margin: float = 0.25
    heavy_second_pass_qtypes: Tuple[str, ...] = ("ocr_text", "location")
    use_easyocr: bool = True
    easyocr_qtypes: Tuple[str, ...] = ("ocr_text",)
    easyocr_langs: Tuple[str, ...] = ("ko", "en")

    # Blend weights by question type
    blend_weights: Dict[str, Dict[str, float]] = None

    # Safety / debug
    save_debug_csv: bool = True
    debug_prediction_name: str = "debug_predictions_colab_h100.csv"

    def __post_init__(self):
        if self.blend_weights is None:
            self.blend_weights = {
                "count":     {"shared": 0.60, "count": 0.28, "text": 0.12, "ocr": 0.00},
                "material":  {"shared": 0.74, "count": 0.00, "text": 0.26, "ocr": 0.00},
                "object":    {"shared": 0.71, "count": 0.08, "text": 0.21, "ocr": 0.00},
                "color":     {"shared": 0.78, "count": 0.00, "text": 0.22, "ocr": 0.00},
                "location":  {"shared": 0.86, "count": 0.00, "text": 0.09, "ocr": 0.05},
                "ocr_text":  {"shared": 0.60, "count": 0.00, "text": 0.05, "ocr": 0.35},
            }

cfg = CFG()
WORK_DIR = Path(cfg.work_dir)
WORK_DIR.mkdir(parents=True, exist_ok=True)
SHARED_ADAPTER_DIR = WORK_DIR / "adapter_shared"
COUNT_ADAPTER_DIR = WORK_DIR / "adapter_count"
TEXT_PRIOR_PATH = WORK_DIR / "text_prior_bundle.pkl"

# -----------------------------
# Utilities
# -----------------------------

LABELS = ["a", "b", "c", "d"]
LABEL2IDX = {x: i for i, x in enumerate(LABELS)}
IDX2LABEL = {i: x for x, i in LABEL2IDX.items()}

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def maybe_enable_perf_flags() -> None:
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass

def print_env() -> None:
    print("=" * 80)
    print("env_name      :", cfg.env_name)
    print("torch         :", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("cuda version  :", torch.version.cuda)
        try:
            print("gpu           :", torch.cuda.get_device_name(0))
        except Exception:
            pass
    print("work_dir      :", str(WORK_DIR.resolve()))
    print("=" * 80)

def auto_choose_attn_impl(default: str = "sdpa") -> str:
    if default == "flash_attention_2":
        try:
            import flash_attn  # noqa: F401
            return "flash_attention_2"
        except Exception:
            return "sdpa"
    return "sdpa"

cfg.attn_implementation = auto_choose_attn_impl(cfg.attn_implementation)

def resolve_data_root(user_root: Optional[str] = None) -> Path:
    candidates = []
    if user_root:
        candidates.append(Path(user_root))
    candidates.extend([
        Path("."),
        Path("/content"),
        Path("/content/data"),
        Path("/content/drive/MyDrive"),
        Path("/mnt/data"),
    ])
    for root in candidates:
        if (root / "train.csv").exists() and (root / "test.csv").exists():
            return root
        # nested competition folder search
        for child in root.glob("**/train.csv"):
            croot = child.parent
            if (croot / "test.csv").exists():
                return croot
    raise FileNotFoundError("Could not locate train.csv/test.csv. Set cfg.data_root explicitly.")

def qtype_from_question(question: str) -> str:
    q = str(question)
    if re.search(r"(몇\s*개|개수|총\s*몇|몇개)", q):
        return "count"
    if re.search(r"(색깔|색)", q):
        return "color"
    if re.search(r"(재질|소재)", q):
        return "material"
    if re.search(r"(브랜드|라벨|문구|글자|로고|적힌|용량|몇\s*ml|몇ml|몇\s*l|몇l|문자|숫자)", q.lower()):
        return "ocr_text"
    if re.search(r"(어디|위치|어느\s*곳|장소)", q):
        return "location"
    return "object"

def normalize_korean_text(s: str) -> str:
    s = str(s).lower().strip()
    s = s.replace("ℓ", "l")
    s = s.replace("㎖", "ml").replace("ml", "ml")
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[^0-9a-zA-Z가-힣mlkgL.%]+", "", s)
    return s

def round_to_multiple(x: int, base: int) -> int:
    return max(base, int(round(x / base) * base))

def resize_image_keep_ratio(img: Image.Image, max_side: int, min_side: int = 448, multiple_of: int = 28) -> Image.Image:
    img = img.convert("RGB")
    w, h = img.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        w = int(w * scale)
        h = int(h * scale)
    if min(w, h) < min_side:
        scale = min_side / min(w, h)
        w = int(w * scale)
        h = int(h * scale)
    w = round_to_multiple(w, multiple_of)
    h = round_to_multiple(h, multiple_of)
    return img.resize((w, h), Image.Resampling.LANCZOS)

def maybe_jpeg_aug(img: Image.Image, quality_low: int = 70, quality_high: int = 95) -> Image.Image:
    if not cfg.jpeg_quality_augment:
        return img
    import io
    q = random.randint(quality_low, quality_high)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=q)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def prompt_hint_by_qtype(qtype: str) -> str:
    if qtype == "count":
        return "개수를 셀 때는 이미지에 실제로 보이는 해당 대상만 세고, 같은 물체를 중복 계산하지 마세요."
    if qtype == "ocr_text":
        return "브랜드, 글자, 숫자, 용량, 라벨, 로고를 먼저 자세히 확인하세요."
    if qtype == "location":
        return "물체의 위치나 영역을 이미지 전체 기준으로 판단하세요."
    if qtype == "material":
        return "재질과 소재를 우선적으로 판단하세요."
    if qtype == "color":
        return "대상 물체의 색상을 정확히 확인하세요."
    return "이미지의 대상 물체 종류를 정확히 파악하세요."

def build_user_prompt(rec: Dict[str, Any]) -> str:
    choices = "\n".join([f"{l}. {rec[l]}" for l in LABELS])
    hint = prompt_hint_by_qtype(rec["qtype"])
    return (
        "당신은 재활용품 이미지 기반 4지선다 VQA 전문가입니다.\n"
        "반드시 소문자 a, b, c, d 중 하나만 답하세요.\n"
        f"{hint}\n\n"
        f"질문: {rec['question']}\n"
        f"선지:\n{choices}\n\n"
        "정답:"
    )

def build_messages_for_record(rec: Dict[str, Any], include_answer: bool = False) -> List[Dict[str, Any]]:
    image_ref = rec.get("image_ref_for_template", "file:///placeholder.jpg")
    messages = [
        {"role": "system", "content": "당신은 재활용품 이미지를 보고 객관식 질문에 답하는 정확한 도우미입니다."},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_ref},
                {"type": "text", "text": build_user_prompt(rec)},
            ],
        },
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": rec["answer"]})
    return messages

def option_permute_record(rec: Dict[str, Any], seed: int) -> Tuple[Dict[str, Any], Dict[str, str]]:
    rng = np.random.default_rng(seed)
    orig_letters = np.array(LABELS)
    perm_old_for_new = list(rng.permutation(orig_letters))  # new a takes old x
    new_rec = copy.deepcopy(rec)
    inverse = {}
    for new_l, old_l in zip(LABELS, perm_old_for_new):
        new_rec[new_l] = rec[old_l]
        inverse[new_l] = old_l
    if "answer" in rec:
        for new_l, old_l in inverse.items():
            if old_l == rec["answer"]:
                new_rec["answer"] = new_l
                break
    return new_rec, inverse

def inverse_map_scores_to_original(permuted_scores: Dict[str, float], inverse: Dict[str, str]) -> Dict[str, float]:
    # inverse[new_letter] = original_letter
    out = {l: -1e9 for l in LABELS}
    for new_l, old_l in inverse.items():
        out[old_l] = float(permuted_scores[new_l])
    return out

def score_margin(scores_vec: np.ndarray) -> float:
    s = np.sort(scores_vec)
    return float(s[-1] - s[-2])

def softmax_np(x: np.ndarray) -> np.ndarray:
    z = x - x.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

# -----------------------------
# Data loading / preparation
# -----------------------------

def load_competition_data(data_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_csv(data_root / "train.csv")
    test_df = pd.read_csv(data_root / "test.csv")
    dev_df = pd.read_csv(data_root / "dev.csv")
    train_df = train_df.dropna(subset=["answer"]).copy().reset_index(drop=True)
    for df in [train_df, test_df, dev_df]:
        df["qtype"] = df["question"].map(qtype_from_question)
    return train_df, test_df, dev_df

def dataframe_to_records(df: pd.DataFrame, data_root: Path, has_answer: bool) -> List[Dict[str, Any]]:
    recs = []
    for row in df.to_dict(orient="records"):
        rec = {
            "id": row["id"],
            "path": row.get("path"),
            "abs_path": str((data_root / row["path"]).resolve()) if pd.notna(row.get("path")) else None,
            "image_ref_for_template": f"file://{(data_root / row['path']).resolve()}" if pd.notna(row.get("path")) else "file:///placeholder.jpg",
            "question": row["question"],
            "a": row["a"],
            "b": row["b"],
            "c": row["c"],
            "d": row["d"],
            "qtype": row.get("qtype", qtype_from_question(row["question"])),
            "source": "train" if has_answer else "test",
        }
        if has_answer:
            rec["answer"] = str(row["answer"]).lower().strip()
        recs.append(rec)
    return recs

def build_dev_pseudo_records(dev_df: pd.DataFrame, data_root: Path, min_conf_shared: float = 0.80, min_conf_count: float = 0.60) -> List[Dict[str, Any]]:
    out = []
    answer_cols = [c for c in dev_df.columns if c.startswith("answer")]
    for row in dev_df.to_dict(orient="records"):
        votes = [str(row[c]).lower().strip() for c in answer_cols if pd.notna(row[c]) and str(row[c]).strip()]
        votes = [v for v in votes if v in LABELS]
        if not votes:
            continue
        counts = Counter(votes)
        top_answer, top_count = counts.most_common(1)[0]
        conf = top_count / len(votes)
        qtype = row.get("qtype", qtype_from_question(row["question"]))
        keep = conf >= min_conf_shared or (qtype == "count" and conf >= min_conf_count)
        if not keep:
            continue
        rec = {
            "id": row["id"],
            "path": row.get("path"),
            "abs_path": str((data_root / row["path"]).resolve()) if pd.notna(row.get("path")) else None,
            "image_ref_for_template": f"file://{(data_root / row['path']).resolve()}" if pd.notna(row.get("path")) else "file:///placeholder.jpg",
            "question": row["question"],
            "a": row["a"],
            "b": row["b"],
            "c": row["c"],
            "d": row["d"],
            "qtype": qtype,
            "answer": top_answer,
            "source": f"dev_pseudo_conf_{conf:.2f}",
            "pseudo_conf": conf,
        }
        out.append(rec)
    return out

def build_trashnet_records() -> List[Dict[str, Any]]:
    """
    Builds public external synthetic QA from TrashNet.
    Synthetic target is material/material-like multiple choice, which is the safest label transfer.
    """
    if not cfg.use_trashnet:
        return []
    print("Loading external public dataset: garythung/trashnet ...")
    ds = load_dataset("garythung/trashnet", split="train")
    # expected features: image, label
    label_col = None
    for c in ds.column_names:
        if c.lower() in ["label", "labels", "class", "classes"]:
            label_col = c
            break
    if label_col is None:
        raise RuntimeError(f"Could not find label column in TrashNet: {ds.column_names}")

    label_feature = ds.features[label_col]
    label_names = None
    if hasattr(label_feature, "names"):
        label_names = list(label_feature.names)

    if label_names is None:
        # fallback: infer from string labels or integers
        uniq = sorted(set(ds[label_col]))
        label_names = [str(x) for x in uniq]

    material_map = {
        "glass": ("유리", "유리병"),
        "metal": ("금속", "금속 캔"),
        "plastic": ("플라스틱", "플라스틱 용기"),
        "paper": ("종이", "종이"),
        "cardboard": ("종이", "종이 상자"),
        "trash": (None, None),
    }
    distractors_by_material = ["유리", "금속", "종이", "플라스틱"]

    recs = []
    for i in range(len(ds)):
        item = ds[i]
        label_idx = item[label_col]
        label_name = label_names[label_idx] if isinstance(label_idx, int) else str(label_idx)
        label_name = str(label_name).lower()
        material, _obj = material_map.get(label_name, (None, None))
        if material is None:
            continue

        options = distractors_by_material.copy()
        if material not in options:
            continue
        random.Random(i + cfg.seed).shuffle(options)
        answer = LABELS[options.index(material)]
        recs.append({
            "id": f"trashnet_{i:05d}",
            "path": None,
            "abs_path": None,
            "image_obj": item["image"].convert("RGB"),
            "image_ref_for_template": "file:///placeholder.jpg",
            "question": "사진 속 재활용품의 재질은 무엇인가요?",
            "a": options[0],
            "b": options[1],
            "c": options[2],
            "d": options[3],
            "qtype": "material",
            "answer": answer,
            "source": "trashnet_public",
        })
    print(f"TrashNet synthetic records: {len(recs):,}")
    return recs

def coarse_taco_group(cat_name: str) -> Optional[str]:
    s = cat_name.lower()
    if any(k in s for k in ["plastic", "pet", "styrofoam", "wrapper", "vinyl"]):
        return "플라스틱"
    if any(k in s for k in ["glass"]):
        return "유리"
    if any(k in s for k in ["metal", "aluminium", "aluminum", "can", "tin"]):
        return "금속"
    if any(k in s for k in ["paper", "cardboard", "carton"]):
        return "종이"
    return None

def build_taco_manual_records() -> List[Dict[str, Any]]:
    """
    Optional hook:
    - Set cfg.use_taco_manual_hook = True
    - Place TACO COCO-format files under cfg.taco_manual_dir
      Expected:
        <taco_dir>/annotations.json  (or annotations/annotations.json)
        <taco_dir>/images/...
    This function then makes coarse count questions from public TACO annotations.
    """
    if not cfg.use_taco_manual_hook or not cfg.taco_manual_dir:
        return []
    taco_dir = Path(cfg.taco_manual_dir)
    if not taco_dir.exists():
        print(f"[TACO] manual dir does not exist: {taco_dir}")
        return []
    ann_candidates = [
        taco_dir / "annotations.json",
        taco_dir / "annotations" / "annotations.json",
    ]
    ann_path = next((p for p in ann_candidates if p.exists()), None)
    if ann_path is None:
        print(f"[TACO] annotations.json not found under {taco_dir}")
        return []

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    cats = {c["id"]: c["name"] for c in coco["categories"]}
    images = {img["id"]: img for img in coco["images"]}
    anns_by_img = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)

    recs = []
    for image_id, anns in anns_by_img.items():
        coarse_counts = defaultdict(int)
        for ann in anns:
            coarse = coarse_taco_group(cats[ann["category_id"]])
            if coarse is not None:
                coarse_counts[coarse] += 1
        if not coarse_counts:
            continue
        image_info = images[image_id]
        file_name = image_info.get("file_name") or image_info.get("path")
        if file_name is None:
            continue
        abs_path = None
        for cand in [taco_dir / file_name, taco_dir / "images" / file_name]:
            if cand.exists():
                abs_path = cand.resolve()
                break
        if abs_path is None:
            continue

        # Create 1 count sample per coarse class in the image.
        for coarse, count in coarse_counts.items():
            count = int(count)
            if count < 1:
                continue
            # competition-like options
            opt_values = sorted(set([max(1, count - 1), count, min(count + 1, count + 2), min(count + 2, count + 3)]))
            while len(opt_values) < 4:
                opt_values.append(opt_values[-1] + 1)
            opt_values = opt_values[:4]
            options_text = [f"{x}개" for x in opt_values]
            rng = random.Random(hash((str(abs_path), coarse, count, cfg.seed)))
            rng.shuffle(options_text)
            correct_text = f"{count}개"
            if correct_text not in options_text:
                options_text[-1] = correct_text
                rng.shuffle(options_text)
            answer = LABELS[options_text.index(correct_text)]
            recs.append({
                "id": f"taco_{image_id}_{coarse}",
                "path": None,
                "abs_path": str(abs_path),
                "image_ref_for_template": f"file://{abs_path}",
                "question": f"사진에 보이는 {coarse} 재질 쓰레기는 몇 개입니까?",
                "a": options_text[0],
                "b": options_text[1],
                "c": options_text[2],
                "d": options_text[3],
                "qtype": "count",
                "answer": answer,
                "source": "taco_manual_public",
            })
    print(f"TACO manual synthetic records: {len(recs):,}")
    return recs

def build_training_pools(train_df: pd.DataFrame, dev_df: pd.DataFrame, data_root: Path) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    train_records = dataframe_to_records(train_df, data_root, has_answer=True)
    pseudo_records = build_dev_pseudo_records(dev_df, data_root, cfg.pseudo_min_conf_shared, cfg.pseudo_min_conf_count) if cfg.use_dev_pseudo else []
    trashnet_records = build_trashnet_records() if cfg.use_trashnet else []
    taco_records = build_taco_manual_records() if cfg.use_taco_manual_hook else []

    shared_pool = []
    for rec in train_records:
        repeat = 1
        if rec["qtype"] == "ocr_text":
            repeat = cfg.upsample_ocr_text
        elif rec["qtype"] == "location":
            repeat = cfg.upsample_location
        elif rec["qtype"] == "color":
            repeat = cfg.upsample_color
        elif rec["qtype"] == "count":
            repeat = cfg.upsample_count
        for _ in range(repeat):
            shared_pool.append(copy.deepcopy(rec))
    shared_pool.extend(copy.deepcopy(r) for r in pseudo_records)
    shared_pool.extend(copy.deepcopy(r) for r in trashnet_records)

    count_pool = [copy.deepcopy(r) for r in train_records if r["qtype"] == "count"]
    count_pool.extend(copy.deepcopy(r) for r in pseudo_records if r["qtype"] == "count")
    count_pool.extend(copy.deepcopy(r) for r in taco_records if r["qtype"] == "count")

    print(f"shared_pool size: {len(shared_pool):,}")
    print(f"count_pool  size: {len(count_pool):,}")
    print(f"pseudo used      : {len(pseudo_records):,}")
    print(f"trashnet used    : {len(trashnet_records):,}")
    print(f"taco used        : {len(taco_records):,}")
    return train_records, shared_pool, count_pool

# -----------------------------
# Processor / model loading
# -----------------------------

def get_bnb_compute_dtype() -> torch.dtype:
    return torch.bfloat16 if cfg.bnb_4bit_compute_dtype == "bfloat16" else torch.float16

def make_quant_config(load_in_4bit: bool):
    if not load_in_4bit:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=get_bnb_compute_dtype(),
        bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
        bnb_4bit_use_double_quant=cfg.bnb_4bit_use_double_quant,
    )

def load_processor():
    # Qwen3-VL processor handles PIL images directly.
    processor = AutoProcessor.from_pretrained(cfg.base_model, trust_remote_code=True)
    if getattr(processor, "tokenizer", None) is not None and processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    return processor

def load_base_model_for_training():
    quant_config = make_quant_config(cfg.train_load_in_4bit)
    kwargs = dict(
        trust_remote_code=True,
        device_map={"": 0} if torch.cuda.is_available() else None,
        quantization_config=quant_config,
        attn_implementation=cfg.attn_implementation,
        low_cpu_mem_usage=True,
    )
    if quant_config is None:
        kwargs["torch_dtype"] = torch.bfloat16 if cfg.use_bf16 else torch.float16

    model = Qwen3VLForConditionalGeneration.from_pretrained(cfg.base_model, **kwargs)
    model.config.use_cache = False
    if cfg.train_load_in_4bit:
        model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        target_modules=list(cfg.lora_target_modules),
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        inference_mode=False,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model

def load_base_model_for_inference(load_in_4bit: bool):
    quant_config = make_quant_config(load_in_4bit)
    kwargs = dict(
        trust_remote_code=True,
        device_map={"": 0} if torch.cuda.is_available() else None,
        quantization_config=quant_config,
        attn_implementation=cfg.attn_implementation,
        low_cpu_mem_usage=True,
    )
    if quant_config is None:
        kwargs["torch_dtype"] = torch.bfloat16 if cfg.use_bf16 else torch.float16

    model = Qwen3VLForConditionalGeneration.from_pretrained(cfg.base_model, **kwargs)
    model.eval()
    return model

# -----------------------------
# Dataset / collator
# -----------------------------

class VQAMCDataset(Dataset):
    def __init__(
        self,
        records: List[Dict[str, Any]],
        processor,
        mode: str = "train",
        option_shuffle_prob: float = 0.0,
        max_side: int = 1024,
    ):
        self.records = records
        self.processor = processor
        self.mode = mode
        self.option_shuffle_prob = option_shuffle_prob
        self.max_side = max_side

    def __len__(self) -> int:
        return len(self.records)

    def _load_image(self, rec: Dict[str, Any]) -> Image.Image:
        if rec.get("image_obj") is not None:
            img = rec["image_obj"].convert("RGB")
        else:
            img = Image.open(rec["abs_path"]).convert("RGB")
        img = resize_image_keep_ratio(img, max_side=self.max_side, min_side=cfg.min_side, multiple_of=cfg.multiple_of)
        if self.mode == "train":
            img = maybe_jpeg_aug(img)
        return img

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        rec = copy.deepcopy(self.records[idx])
        if self.mode == "train" and random.random() < self.option_shuffle_prob:
            rec, _ = option_permute_record(rec, seed=cfg.seed + idx + random.randint(0, 10_000_000))
        image = self._load_image(rec)
        prefix_messages = build_messages_for_record(rec, include_answer=False)
        full_messages = build_messages_for_record(rec, include_answer=("answer" in rec))
        prefix_text = self.processor.apply_chat_template(
            prefix_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        if "answer" in rec:
            full_text = self.processor.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        else:
            full_text = None

        return {
            "id": rec["id"],
            "image": image,
            "prefix_text": prefix_text,
            "full_text": full_text,
            "answer": rec.get("answer"),
            "record": rec,
        }

class TrainCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        images = [x["image"] for x in batch]
        prefix_texts = [x["prefix_text"] for x in batch]
        full_texts = [x["full_text"] for x in batch]

        full_inputs = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        prefix_inputs = self.processor(
            text=prefix_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        full_inputs.pop("token_type_ids", None)
        prefix_inputs.pop("token_type_ids", None)

        labels = full_inputs["input_ids"].clone()
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        prefix_lens = prefix_inputs["attention_mask"].sum(dim=1).tolist()
        for i, plen in enumerate(prefix_lens):
            labels[i, :int(plen)] = -100

        full_inputs["labels"] = labels
        return full_inputs

class InferCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch: List[Dict[str, Any]]) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
        images = [x["image"] for x in batch]
        prefix_texts = [x["prefix_text"] for x in batch]
        inputs = self.processor(
            text=prefix_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        inputs.pop("token_type_ids", None)
        metas = [{"id": x["id"], "record": x["record"]} for x in batch]
        return inputs, metas

# -----------------------------
# Training
# -----------------------------

def save_json(path: Path, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def train_adapter(records: List[Dict[str, Any]], adapter_dir: Path, num_epochs: int, batch_size: int, lr: float, max_side: int) -> None:
    if adapter_dir.exists() and any(adapter_dir.glob("adapter_model*")) and not cfg.force_retrain:
        print(f"[skip] adapter exists: {adapter_dir}")
        return

    adapter_dir.mkdir(parents=True, exist_ok=True)
    processor = load_processor()
    model = load_base_model_for_training()

    train_ds = VQAMCDataset(
        records=records,
        processor=processor,
        mode="train",
        option_shuffle_prob=cfg.option_shuffle_prob,
        max_side=max_side,
    )
    loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        collate_fn=TrainCollator(processor),
    )

    num_update_steps_per_epoch = max(1, math.ceil(len(loader) / cfg.gradient_accumulation_steps))
    total_steps = num_epochs * num_update_steps_per_epoch
    warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))

    try:
        import bitsandbytes as bnb
        optimizer = bnb.optim.PagedAdamW8bit(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr,
            betas=(0.9, 0.95),
            weight_decay=cfg.weight_decay,
        )
    except Exception:
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr,
            betas=(0.9, 0.95),
            weight_decay=cfg.weight_decay,
        )

    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    device = torch.device(cfg.device)
    autocast_dtype = torch.bfloat16 if cfg.use_bf16 else torch.float16

    global_step = 0
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(num_epochs):
        running_loss = 0.0
        step_in_epoch = 0
        t0 = time.time()

        for step, batch in enumerate(loader):
            step_in_epoch += 1
            batch = {k: v.to(device) for k, v in batch.items()}

            with torch.autocast(device_type=("cuda" if torch.cuda.is_available() else "cpu"), dtype=autocast_dtype, enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = outputs.loss / cfg.gradient_accumulation_steps

            loss.backward()
            running_loss += float(loss.item()) * cfg.gradient_accumulation_steps

            if (step + 1) % cfg.gradient_accumulation_steps == 0 or (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if global_step % cfg.logging_steps == 0 or global_step == 1:
                    elapsed = time.time() - t0
                    avg_loss = running_loss / max(1, step_in_epoch)
                    print(
                        f"[{adapter_dir.name}] epoch={epoch+1}/{num_epochs} "
                        f"global_step={global_step}/{total_steps} "
                        f"loss={avg_loss:.4f} "
                        f"lr={scheduler.get_last_lr()[0]:.3e} "
                        f"time={elapsed/60:.1f}m"
                    )

        if cfg.save_every_epoch:
            epoch_dir = adapter_dir / f"epoch_{epoch+1}"
            epoch_dir.mkdir(exist_ok=True, parents=True)
            model.save_pretrained(epoch_dir)
            processor.save_pretrained(epoch_dir)

    model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)
    save_json(adapter_dir / "training_config.json", asdict(cfg))

    del model
    del processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# -----------------------------
# Text prior
# -----------------------------

def build_text_prior_feature_from_row(row: Dict[str, Any]) -> str:
    return (
        f"[QTYPE]{row.get('qtype', qtype_from_question(row['question']))}[/QTYPE] "
        f"[Q]{row['question']}[/Q] "
        f"[A]{row['a']}[/A] [B]{row['b']}[/B] [C]{row['c']}[/C] [D]{row['d']}[/D]"
    )

def fit_text_prior(train_df: pd.DataFrame):
    if not cfg.use_text_prior:
        return None
    import pickle

    x = [build_text_prior_feature_from_row(r) for r in train_df.to_dict(orient="records")]
    y = train_df["answer"].astype(str).str.lower().tolist()

    vectorizer = TfidfVectorizer(
        analyzer=cfg.tfidf_analyzer,
        ngram_range=cfg.tfidf_ngram_range,
        min_df=cfg.tfidf_min_df,
        lowercase=False,
        sublinear_tf=True,
    )
    X = vectorizer.fit_transform(x)
    clf = LogisticRegression(
        C=cfg.text_prior_c,
        max_iter=cfg.text_prior_max_iter,
        solver="lbfgs",
        multi_class="multinomial",
        random_state=cfg.seed,
    )
    clf.fit(X, y)
    bundle = {"vectorizer": vectorizer, "clf": clf}
    with open(TEXT_PRIOR_PATH, "wb") as f:
        pickle.dump(bundle, f)
    print(f"Saved text prior: {TEXT_PRIOR_PATH}")
    return bundle

def load_text_prior():
    if not cfg.use_text_prior or not TEXT_PRIOR_PATH.exists():
        return None
    import pickle
    with open(TEXT_PRIOR_PATH, "rb") as f:
        bundle = pickle.load(f)
    return bundle

def predict_text_prior(bundle, df_or_records: List[Dict[str, Any]]) -> np.ndarray:
    if bundle is None:
        return np.zeros((len(df_or_records), 4), dtype=np.float32)
    feats = [build_text_prior_feature_from_row(r) for r in df_or_records]
    X = bundle["vectorizer"].transform(feats)
    proba = bundle["clf"].predict_proba(X)
    # reorder to a,b,c,d
    class_to_idx = {c: i for i, c in enumerate(bundle["clf"].classes_)}
    out = np.zeros((len(df_or_records), 4), dtype=np.float32)
    for j, label in enumerate(LABELS):
        out[:, j] = proba[:, class_to_idx[label]]
    return out

# -----------------------------
# Multiple-choice logprob scoring
# -----------------------------

def get_letter_token_ids(processor) -> Dict[str, int]:
    out = {}
    for label in LABELS:
        ids = processor.tokenizer(label, add_special_tokens=False).input_ids
        if len(ids) != 1:
            raise ValueError(f"Expected single token for label `{label}`, got ids={ids}")
        out[label] = ids[0]
    return out

@torch.no_grad()
def predict_letter_scores(
    model,
    processor,
    records: List[Dict[str, Any]],
    adapter_name: Optional[str] = None,
    num_permutations: int = 1,
    max_side: int = 1024,
    batch_size: int = 1,
    desc: str = "predict",
) -> np.ndarray:
    if hasattr(model, "set_adapter") and adapter_name is not None:
        model.set_adapter(adapter_name)
    model.eval()

    letter_token_ids = get_letter_token_ids(processor)
    final_scores = np.zeros((len(records), 4), dtype=np.float32)

    device = torch.device(cfg.device)
    autocast_dtype = torch.bfloat16 if cfg.use_bf16 else torch.float16

    for perm_idx in range(num_permutations):
        permuted_items = []
        inverse_maps = []
        for i, rec in enumerate(records):
            if perm_idx == 0:
                perm_rec = copy.deepcopy(rec)
                inverse = {l: l for l in LABELS}
            else:
                seed = int(hashlib.md5(f"{rec['id']}_{perm_idx}_{cfg.seed}".encode()).hexdigest()[:8], 16)
                perm_rec, inverse = option_permute_record(rec, seed=seed)
            inverse_maps.append(inverse)
            permuted_items.append(perm_rec)

        infer_ds = VQAMCDataset(
            records=permuted_items,
            processor=processor,
            mode="infer",
            option_shuffle_prob=0.0,
            max_side=max_side,
        )
        loader = DataLoader(
            infer_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=cfg.pin_memory,
            collate_fn=InferCollator(processor),
        )

        cursor = 0
        for batch_inputs, metas in loader:
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            with torch.autocast(device_type=("cuda" if torch.cuda.is_available() else "cpu"), dtype=autocast_dtype, enabled=torch.cuda.is_available()):
                outputs = model(**batch_inputs)
                logits = outputs.logits

            attn = batch_inputs["attention_mask"]
            batch_size_real = attn.shape[0]
            for i in range(batch_size_real):
                last_pos = int(attn[i].sum().item()) - 1
                lp = F.log_softmax(logits[i, last_pos, :], dim=-1)
                perm_scores = {label: float(lp[token_id].item()) for label, token_id in letter_token_ids.items()}
                mapped = inverse_map_scores_to_original(perm_scores, inverse_maps[cursor + i])
                final_scores[cursor + i] += np.array([mapped[l] for l in LABELS], dtype=np.float32)
            cursor += batch_size_real

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_scores /= float(num_permutations)
    return final_scores

# -----------------------------
# EasyOCR booster
# -----------------------------

def build_easyocr_reader():
    if not cfg.use_easyocr:
        return None
    import easyocr
    return easyocr.Reader(list(cfg.easyocr_langs), gpu=torch.cuda.is_available())

def token_overlap_ratio(a: str, b: str) -> float:
    toks_a = re.findall(r"[0-9a-zA-Z가-힣]+", a)
    toks_b = set(re.findall(r"[0-9a-zA-Z가-힣]+", b))
    if not toks_a:
        return 0.0
    hit = sum(1 for t in toks_a if t in toks_b)
    return hit / max(1, len(toks_a))

def easyocr_boost_for_record(reader, rec: Dict[str, Any]) -> np.ndarray:
    if reader is None:
        return np.zeros(4, dtype=np.float32)
    if rec["qtype"] not in cfg.easyocr_qtypes:
        return np.zeros(4, dtype=np.float32)

    if rec.get("image_obj") is not None:
        img = rec["image_obj"].convert("RGB")
    else:
        img = Image.open(rec["abs_path"]).convert("RGB")
    img = resize_image_keep_ratio(img, max_side=cfg.heavy_max_side, min_side=cfg.min_side, multiple_of=cfg.multiple_of)
    txts = reader.readtext(np.array(img), detail=0, paragraph=True)
    joined_raw = " ".join(map(str, txts))
    joined = normalize_korean_text(joined_raw)

    boosts = np.zeros(4, dtype=np.float32)
    for j, label in enumerate(LABELS):
        opt_raw = str(rec[label])
        opt = normalize_korean_text(opt_raw)
        boost = 0.0
        if opt and opt in joined:
            boost += 2.0
        for num in re.findall(r"\d+(?:\.\d+)?", opt_raw.lower()):
            if num in joined_raw.lower():
                boost += 0.8
        if any(u in opt for u in ["ml", "l", "g", "kg", "%"]) and any(u in joined for u in ["ml", "l", "g", "kg", "%"]):
            boost += 0.4
        boost += 0.5 * token_overlap_ratio(opt_raw.lower(), joined_raw.lower())
        boosts[j] = boost
    return boosts

def predict_easyocr_boosts(records: List[Dict[str, Any]]) -> np.ndarray:
    if not cfg.use_easyocr:
        return np.zeros((len(records), 4), dtype=np.float32)
    reader = build_easyocr_reader()
    out = np.zeros((len(records), 4), dtype=np.float32)
    for i, rec in enumerate(records):
        if rec["qtype"] in cfg.easyocr_qtypes:
            out[i] = easyocr_boost_for_record(reader, rec)
    return out

# -----------------------------
# Adapter loading for inference
# -----------------------------

def load_inference_model_with_adapters() -> Tuple[Any, Any]:
    processor = load_processor()
    base = load_base_model_for_inference(cfg.infer_load_in_4bit)

    if not SHARED_ADAPTER_DIR.exists():
        raise FileNotFoundError(f"Shared adapter not found: {SHARED_ADAPTER_DIR}")

    try:
        model = PeftModel.from_pretrained(base, str(SHARED_ADAPTER_DIR), adapter_name="shared")
    except TypeError:
        model = PeftModel.from_pretrained(base, str(SHARED_ADAPTER_DIR))

    if cfg.do_train_count_expert and COUNT_ADAPTER_DIR.exists():
        try:
            model.load_adapter(str(COUNT_ADAPTER_DIR), adapter_name="count")
        except TypeError:
            try:
                model.load_adapter(str(COUNT_ADAPTER_DIR))
            except Exception:
                pass
        except Exception:
            pass

    model.eval()
    return model, processor

# -----------------------------
# Blending / submission
# -----------------------------

def blend_component_scores(
    records: List[Dict[str, Any]],
    shared_scores_logit: np.ndarray,
    count_scores_logit: Optional[np.ndarray],
    text_probs: Optional[np.ndarray],
    ocr_boosts: Optional[np.ndarray],
) -> Tuple[List[str], pd.DataFrame]:
    shared_probs = softmax_np(shared_scores_logit)
    count_probs = softmax_np(count_scores_logit) if count_scores_logit is not None else None
    text_probs = text_probs if text_probs is not None else None

    final_preds = []
    debug_rows = []

    for i, rec in enumerate(records):
        qtype = rec["qtype"]
        weights = cfg.blend_weights[qtype]

        parts = []
        part_weights = []

        parts.append(shared_probs[i])
        part_weights.append(weights["shared"])

        if count_probs is not None and (weights["count"] > 0):
            parts.append(count_probs[i])
            part_weights.append(weights["count"])

        if text_probs is not None and (weights["text"] > 0):
            parts.append(text_probs[i])
            part_weights.append(weights["text"])

        if ocr_boosts is not None and (weights["ocr"] > 0) and np.any(ocr_boosts[i] > 0):
            ocr_part = softmax_np(ocr_boosts[i][None, :])[0]
            parts.append(ocr_part)
            part_weights.append(weights["ocr"])

        norm = max(1e-8, float(sum(part_weights)))
        score = sum(w * p for w, p in zip(part_weights, parts)) / norm

        margin = score_margin(score)
        pred = LABELS[int(score.argmax())]

        debug_rows.append({
            "id": rec["id"],
            "qtype": qtype,
            "pred": pred,
            "margin": margin,
            "shared_pred": LABELS[int(shared_probs[i].argmax())],
            "count_pred": LABELS[int(count_probs[i].argmax())] if count_probs is not None else None,
            "text_pred": LABELS[int(text_probs[i].argmax())] if text_probs is not None else None,
            "ocr_pred": LABELS[int(ocr_boosts[i].argmax())] if (ocr_boosts is not None and np.any(ocr_boosts[i] > 0)) else None,
            "score_a": float(score[0]),
            "score_b": float(score[1]),
            "score_c": float(score[2]),
            "score_d": float(score[3]),
            "a": rec["a"], "b": rec["b"], "c": rec["c"], "d": rec["d"],
            "question": rec["question"],
        })
        final_preds.append(pred)

    return final_preds, pd.DataFrame(debug_rows)

def heavy_second_pass_refine(
    model,
    processor,
    records: List[Dict[str, Any]],
    base_preds: List[str],
    debug_df: pd.DataFrame,
    shared_scores_logit: np.ndarray,
) -> Tuple[List[str], pd.DataFrame]:
    """
    Optional second pass:
    - only for low-margin samples or OCR/location qtypes
    - uses higher-res shared adapter scores
    """
    if not cfg.do_heavy_second_pass:
        return base_preds, debug_df

    target_indices = []
    for i, rec in enumerate(records):
        qtype = rec["qtype"]
        margin = float(debug_df.iloc[i]["margin"])
        if margin < cfg.heavy_second_pass_margin or qtype in cfg.heavy_second_pass_qtypes:
            target_indices.append(i)

    if not target_indices:
        return base_preds, debug_df

    sub_records = [records[i] for i in target_indices]
    hi_scores = predict_letter_scores(
        model=model,
        processor=processor,
        records=sub_records,
        adapter_name="shared",
        num_permutations=max(1, cfg.infer_num_permutations),
        max_side=cfg.heavy_max_side,
        batch_size=cfg.infer_batch_size,
        desc="heavy_second_pass",
    )
    hi_probs = softmax_np(hi_scores)

    new_preds = list(base_preds)
    for local_i, global_i in enumerate(target_indices):
        old = debug_df.loc[global_i, ["score_a", "score_b", "score_c", "score_d"]].to_numpy(dtype=np.float32)
        mixed = 0.60 * old + 0.40 * hi_probs[local_i]
        pred = LABELS[int(mixed.argmax())]
        new_preds[global_i] = pred
        debug_df.loc[global_i, "pred"] = pred
        debug_df.loc[global_i, "margin"] = float(score_margin(mixed))
        debug_df.loc[global_i, ["score_a", "score_b", "score_c", "score_d"]] = mixed.tolist()
        debug_df.loc[global_i, "heavy_refined"] = True

    return new_preds, debug_df

# -----------------------------
# Main
# -----------------------------

def main():
    seed_everything(cfg.seed)
    maybe_enable_perf_flags()
    print_env()

    data_root = resolve_data_root(cfg.data_root)
    print(f"data_root = {data_root}")

    train_df, test_df, dev_df = load_competition_data(data_root)
    print("train shape:", train_df.shape)
    print("test  shape:", test_df.shape)
    print("dev   shape:", dev_df.shape)
    print("train qtype dist:\n", train_df["qtype"].value_counts(normalize=True).round(3))
    print("test  qtype dist:\n", test_df["qtype"].value_counts(normalize=True).round(3))
    print("dev   qtype dist:\n", dev_df["qtype"].value_counts(normalize=True).round(3))

    train_records_base, shared_pool, count_pool = build_training_pools(train_df, dev_df, data_root)
    test_records = dataframe_to_records(test_df, data_root, has_answer=False)

    if cfg.use_text_prior:
        fit_text_prior(train_df)

    if cfg.do_train and cfg.do_train_shared:
        print("\n=== Training shared adapter ===")
        train_adapter(
            records=shared_pool,
            adapter_dir=SHARED_ADAPTER_DIR,
            num_epochs=cfg.shared_epochs,
            batch_size=cfg.shared_train_batch_size,
            lr=cfg.lr_shared,
            max_side=cfg.train_max_side,
        )

    if cfg.do_train and cfg.do_train_count_expert and len(count_pool) > 0:
        print("\n=== Training count expert adapter ===")
        train_adapter(
            records=count_pool,
            adapter_dir=COUNT_ADAPTER_DIR,
            num_epochs=cfg.count_epochs,
            batch_size=cfg.count_train_batch_size,
            lr=cfg.lr_count,
            max_side=cfg.train_max_side,
        )

    if not cfg.do_infer:
        print("Training finished. Inference disabled.")
        return

    print("\n=== Loading inference model ===")
    model, processor = load_inference_model_with_adapters()

    print("\n=== Shared adapter inference ===")
    shared_scores = predict_letter_scores(
        model=model,
        processor=processor,
        records=test_records,
        adapter_name="shared",
        num_permutations=cfg.infer_num_permutations,
        max_side=cfg.infer_max_side,
        batch_size=cfg.infer_batch_size,
        desc="shared",
    )

    count_scores = None
    if cfg.do_train_count_expert and COUNT_ADAPTER_DIR.exists():
        print("\n=== Count adapter inference ===")
        count_scores = predict_letter_scores(
            model=model,
            processor=processor,
            records=test_records,
            adapter_name="count",
            num_permutations=max(1, cfg.infer_num_permutations),
            max_side=cfg.infer_max_side,
            batch_size=cfg.infer_batch_size,
            desc="count",
        )

    print("\n=== Text prior inference ===")
    text_prior_bundle = load_text_prior()
    text_probs = predict_text_prior(text_prior_bundle, test_records) if text_prior_bundle is not None else None

    print("\n=== EasyOCR boosts ===")
    ocr_boosts = predict_easyocr_boosts(test_records) if cfg.use_easyocr else None

    print("\n=== Blending ===")
    final_preds, debug_df = blend_component_scores(
        records=test_records,
        shared_scores_logit=shared_scores,
        count_scores_logit=count_scores,
        text_probs=text_probs,
        ocr_boosts=ocr_boosts,
    )

    print("\n=== Heavy second pass ===")
    final_preds, debug_df = heavy_second_pass_refine(
        model=model,
        processor=processor,
        records=test_records,
        base_preds=final_preds,
        debug_df=debug_df,
        shared_scores_logit=shared_scores,
    )

    submission = pd.DataFrame({
        "id": test_df["id"].tolist(),
        "answer": final_preds,
    })
    sub_path = WORK_DIR / cfg.submission_name
    submission.to_csv(sub_path, index=False, encoding="utf-8-sig")
    print(f"Saved submission: {sub_path}")

    if cfg.save_debug_csv:
        debug_path = WORK_DIR / cfg.debug_prediction_name
        debug_df.to_csv(debug_path, index=False, encoding="utf-8-sig")
        print(f"Saved debug csv: {debug_path}")

    # quick summary
    print(submission["answer"].value_counts())



In [ ]:
# ===== Optional config overrides =====
cfg.data_root = None   # auto search. Example: "/content" or "/content/my_competition_dir"
cfg.use_trashnet = True
cfg.use_taco_manual_hook = False
cfg.taco_manual_dir = None  # Example: "/content/external/taco"

cfg.force_retrain = False

# Faster smoke test
# cfg.shared_epochs = 1
# cfg.count_epochs = 1
# cfg.infer_num_permutations = 1
# cfg.do_heavy_second_pass = False
# cfg.use_easyocr = False

print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))

## 4) 실행

끝나면 `work_qwen3vl_colab_h100/submission_qwen3vl_colab_h100.csv`가 생성된다.


In [ ]:
main()